In [13]:
from PIL import Image

In [2]:
INSTRUCTION = ("You are a robot navigating an enclosed space."
" Your goal is to navigate to the correct object based on the user's commands. You were given the following task by the"
" user '{TASK}'. Currently, you are facing a scene represented by the given image. Reason about what you are seeing,"
" comparing what you know about the task (given the user commands) and the given scene. For example, if the task is"
" 'Navigate to the black leather sofa near a lampstand' your reasoning process will be"
" 'I'm currently observing a brown sofa which is different than"
" black, making it unlikely to be the target sofa. Moreover, there"
" is no lampstand near it, only a rug and a window' etc. If there"
" are distortions or artifact, do not focus on them, focus on the"
" object at hand. At the end of the reasoning process, evaluate"
" how well the provided image aligns with the user's task. Assign"
" a confidence score based on the following scale: - 0: You are"
" certain the image DOES NOT match the task. - 1: You are unsure"
" whether the image matches the task or not. - 2: You are certain"
" the image DOES match the task. Provide a concise reasoning"
" (under 100 words) and strictly follow this output format:\n"
"<motivation>Your reasoning here</motivation><score>0, 1, or 2</score>")

MODEL_NAME = "e-zorzi/Qwen2.5-VL-7B-Instruct-tuned-final"
DATASET_NAME = "reasoning-augmentation/rubrics"

In [6]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME)
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'instruction', 'reasoning', 'score', 'id', 'original_id', 'reasoning_type', 'rubrics'],
        num_rows: 604
    })
})

In [10]:
NUM_ROW = 555

row = dataset['train'][NUM_ROW]
task = row['instruction']
image = row['image']
reasoning = row['reasoning']
score = row['score']

image.save("./image.png")

image, task, reasoning, score


(<PIL.PngImagePlugin.PngImageFile image mode=RGB size=512x512>,
 'Navigate to the bed with tufted headboard',
 'The bed in the image has a red tufted headboard, which matches the description in the task. The room appears to be a bedroom, and the bed is the central piece of furniture, indicating it is likely the intended target. There are no other beds visible in the room, so this one is the most likely candidate.',
 2)

In [11]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info  # pip install qwen-vl-utils
import torch

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"  # swap for your checkpoint (local path or HF repo id)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,   # use torch.float16 if bf16 isn't supported
    device_map="cuda:0",
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)


def generate(prompt, image_path=None, image_url=None, max_new_tokens=512):
    content = []
    if image_path:
        content.append({"type": "image", "image": f"file://{image_path}"})
    elif image_url:
        content.append({"type": "image", "image": image_url})
    content.append({"type": "text", "text": prompt})

    messages = [{"role": "user", "content": content}]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    # strip prompt tokens, keep only newly generated ones
    trimmed = [
        out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)
    ]
    return processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]


if __name__ == "__main__":
    examples = [
        {"prompt": INSTRUCTION.format(TASK=task), "image_path": "./image.png"},
    ]

    for ex in examples:
        print("=" * 80)
        print(f"PROMPT: {ex['prompt']}")
        print("-" * 80)
        print(generate(**ex))

Loading weights: 100%|██████████| 824/824 [00:00<00:00, 1646.34it/s]


PROMPT: You are a robot navigating an enclosed space. Your goal is to navigate to the correct object based on the user's commands. You were given the following task by the user 'Navigate to the bed with tufted headboard'. Currently, you are facing a scene represented by the given image. Reason about what you are seeing, comparing what you know about the task (given the user commands) and the given scene. For example, if the task is 'Navigate to the black leather sofa near a lampstand' your reasoning process will be 'I'm currently observing a brown sofa which is different than black, making it unlikely to be the target sofa. Moreover, there is no lampstand near it, only a rug and a window' etc. If there are distortions or artifact, do not focus on them, focus on the object at hand. At the end of the reasoning process, evaluate how well the provided image aligns with the user's task. Assign a confidence score based on the following scale: - 0: You are certain the image DOES NOT match the